# MLB 2026 Win Prediction — Data Preparation Notes

## 1. Data Sources
- `PlayerTeamsAll_2026.csv` — 2026 projected rosters with roles and weights
- `CareerHittingStatsAlltime.csv` — Career hitting stats for all players
- `CareerFieldingStatsAlltime.csv` — Career fielding stats for all players
- `CareerPitchingStatsAlltime.csv` — Career pitching stats for all players

---

## 2. Historical Team Filtering
Removed all rows belonging to defunct, historical, or Negro League franchises
from the career stats dataframes. This includes teams like the Homestead Grays,
Kansas City Monarchs, Brooklyn Dodgers, and other pre-modern or relocated franchises.
Null team names were also dropped in the same step.

**Affected dataframes:** `career_hitting`, `career_fielding`, `career_pitching`

---

## 3. Team Code Standardization
Career stats use full team names (e.g. "Los Angeles Dodgers") while roster data
uses abbreviations (e.g. "LAD"). A `match()` function was written to convert full
team names to their standard 2-3 letter codes by:

- Building abbreviation candidates from initials (e.g. "LAD", "LA", "L")
- Handling edge cases explicitly:
  - Arizona Diamondbacks → `ARI`
  - Washington Nationals / Montreal Expos → `WSN`
  - Florida Marlins → `MIA`
  - Chicago White Sox → `CHW`
  - All Angels franchises → `LAA`
  - All Tampa Bay franchises → `TBR`
  - St. Louis Cardinals → `STL`

Rows that could not be matched were tagged `"No Match"` for review.

**Result:** Team code column added to `career_hitting`, `career_fielding`, `career_pitching`

---

## 4. Column Renaming — Inference to Training Alignment
Raw API column names were mapped to match the training data schema using
rename dictionaries:

- **Offense:** e.g. `atBats → AB`, `homeRuns → HR`, `baseOnBalls → BB`
- **Pitching:** e.g. `P_wins → W_pitch`, `P_era → ERA_pitch`, `P_inningsPitched → IP_pitch`

---

## 5. Derived Columns
Columns present in inference data but missing from training data were computed:

### Hitting
| Column | Formula |
|--------|---------|
| `totalBases` | Singles + 2×2B + 3×3B + 4×HR |
| `stolenBasePercentage` | SB / (SB + CS) |
| `caughtStealingPercentage` | CS / (SB + CS) |
| `atBatsPerHomeRun` | AB / HR |
| `plateAppearances` | AB + BB (approximate) |
| `babip` | (H - HR) / (AB - SO - HR) (approximate, missing SF) |

### Pitching
| Column | Formula |
|--------|---------|
| `P_strikeoutWalkRatio` | SO / BB |
| `P_winPercentage` | W / (W + L) |
| `P_strikeoutsPer9Inn` | (SO / IP) × 9 |
| `P_walksPer9Inn` | (BB / IP) × 9 |
| `P_hitsPer9Inn` | (H / IP) × 9 |
| `P_homeRunsPer9` | (HR / IP) × 9 |
| `P_runsScoredPer9` | (R / IP) × 9 |

---

## 6. Team-Level Projection — Hitting (`aggregateHitting`)
Player-level stats aggregated to team-level projections using a
weighted average approach driven by `Player_Weight` (projected playing time).

**Tally columns** (counting stats like AB, H, HR):
- Normalized to per-game rates: `stat / G`
- Weighted: `rate × Player_Weight`
- Summed by team, scaled by `162 games × 10 slots`
- Mean-shifted to match historical league averages

**Rate columns** (AVG, OBP, SLG, OPS, fielding_pct etc.):
- Weighted average only: `stat × Player_Weight`, summed by team
- No season scaling applied

---

## 7. Team-Level Projection — Pitching (`aggregatePitching`)
Same weighted approach as hitting but normalized by innings pitched (`IP_pitch`)
instead of games played.

**Tally columns** split into three groups:
- `starter_tallies` — GS, CG, SHO, IP
- `closer_tallies` — SV, SVO (handled separately, see below)
- `general_tallies` — W, L, G, H, R, ER, HR, HB, BB, SO

**Rate columns:** ERA, WHIP, AVG_pitch and all per-9 / ratio stats

### Save Column Weighting
Saves and save opportunities projected using role-based weights:
- If both Closer and Setup Men present: **70% Closer / 30% Setup Men**
- If only Closers: 100% split evenly among closers
- If only Setup Men: 100% split evenly among setup men

---

## 8. Mean Shifting
After scaling, all tally projections were shifted so the league-wide mean
matches the historical training data mean. This corrects for systematic
over/under-estimation in the projection methodology without distorting
the relative spread between teams.
```python
shift = historical_mean - projected_mean
projected_column = projected_column + shift
```

In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')
data_dir = os.path.join(os.getcwd(), '..', 'data')
scripts_dir = os.path.join(os.getcwd(), '..', 'scripts')


## **Team Filtering & Code Standardization**
Loaded career hitting, fielding, and pitching stats and removed all defunct/historic
franchises (Negro League, relocated, and pre-modern teams) as well as null team names.
Applied a `match()` function to convert full team names to standard abbreviations,
with explicit handling for edge cases like the Diamondbacks, Angels, and Tampa Bay franchises.
Rows with no matching code are tagged `"No Match"` for review.

In [103]:
historic_teams = [
    "Homestead Grays",
    "Kansas City Athletics",
    "Brooklyn Superbas",
    "Washington Senators",
    "Jacksonville Red Caps",
    "Newark Eagles",
    "Chicago American Giants",
    "St. Louis Stars",
    "Brooklyn Dodgers",
    "St. Louis Browns",
    "Indianapolis ABC's",
    "Memphis Red Sox",
    "New York Black Yankees",
    "Birmingham Black Barons",
    "Kansas City Monarchs",
    "Louisville Black Caps",
    "Oakland Athletics",
    "New York Cubans",
    "Washington Elite Giants",
    "Brooklyn Robins",
    "New York Giants",
    "Washington Pilots",
    "New York Highlanders",
    "Chicago Giants",
    "Indianapolis Athletics",
    "Nashville Elite Giants",
    "Akron Grays",
    "Montgomery Grey Sox",
    "Little Rock Grays",
    "Indianapolis Clowns",
    "Toledo Tigers",
    "Toledo-Indianapolis Crawfords",
    "Louisville White Sox",
    "St. Louis-New Orleans Stars",
    "Cuban Stars West",
    "Hilldale Daisies",
    "Chicago Orphans",
    "Brooklyn Royal Giants",
    "St. Louis Giants",
    "New York Lincoln Giants",
    "Harrisburg Giants",
    "Monroe Monarchs",
    "Washington Black Senators",
    "Brooklyn Eagles",
    "Cuban Stars East",
    "Newark Dodgers",
    "Pollock's Cuban Stars",
    "Dayton Marcos",
    "Newark Browns",
    "Toledo Crawfords",
    "Harrisburgh-St. Louis Stars",
    "Todelo Tigers",
     'Detroit Wolves',
    'Cleveland Red Sox'
]
non_current_teams = [
    "Philadelphia Stars",
    "Cleveland Indians",
    "Detroit Stars",
    "Boston Braves",
    "Philadelphia Bacharach Giants",
    "Cleveland Bears",
    "California Angels",
    "Philadelphia Athletics",
    "Houston Colt 45's",
    "Baltimore Elite Giants",
    "Montreal Expos",
    "Cleveland Buckeyes",
    "Anaheim Angels",
    "Cincinnati-Indianapolis Clowns",
    "Cleveland Blues",
    "Boston Beaneaters",
    "Cleveland Bronchos",
    "Tampa Bay Devil Rays",
    "Boston Doves",
    "Cincinnati Redlegs",
    "Florida Marlins",
    "Boston Bees",
    "Atlanta Black Crackers",
    "Cleveland Naps",
    "Milwaukee Braves",
    "Atlantic City Bacharach Giants",
    "Baltimore Black Sox",
    "Pittsburgh Keystones",
    "Pittsburgh Crawfords",
    "Cole's American Giants",
    "Cincinnati Clowns",
    "Cincinnati Buckeyes",
    "Milwaukee Bears",
    "Boston Americans",
    "Cincinnati Tigers",
    "Cleveland Tate Stars",
    "Cincinnati Cuban Stars",
    "Columbus Buckeyes",
    "Boston Rustlers",
    "Baltimore Sox",
    "Columbus Elite Giants",
    "Columbus Blue Birds",
    "Cleveland Stars",
    "Seattle Pilots"
]
historic_teams+=non_current_teams
rosters=pd.read_csv(os.path.join(data_dir,'PlayerTeamsAll_2026.csv'))
rosters['first_name']=rosters['Name'].apply(lambda x: x.split()[0])
rosters['last_name']=rosters['Name'].apply(lambda x: " ".join(x.split()[1:]))
career_hitting=pd.read_csv(os.path.join(data_dir,'CareerHittingStatsAlltime.csv'))
career_fielding=pd.read_csv(os.path.join(data_dir,'CareerFieldingStatsAlltime.csv'))
career_pitching=pd.read_csv(os.path.join(data_dir,'CareerPitchingStatsAlltime.csv'))
rosters

# 1. Filter out historic teams AND remove null values in one step
career_hitting = career_hitting[
    (~career_hitting['team_name'].isin(historic_teams)) & (career_hitting['team_name'].notna())
]

career_fielding = career_fielding[
    (~career_fielding['team_name'].isin(historic_teams)) & (career_fielding['team_name'].notna())
]

career_pitching = career_pitching[
    (~career_pitching['team_name'].isin(historic_teams)) & (career_pitching['team_name'].notna())
]

def match(s1,codes):
    L=[]
    one=s1.split(' ')
    id=''
    for item in one:
        id+=item[0].upper()
        L.append(id)
    L.append(one[0][0:3].upper())
    if s1=='Arizona Diamondbacks':
        return 'ARI'
    elif s1 in ['Washington Nationals','Montreal Expos']:
        return 'WSN'
    elif s1=='St. Louis Cardinals':
        return 'STL'
    elif s1=='Chicago Cubs':
        return 'CHC'
    elif s1=='Toronto Blue Jays':
        return 'TOR'
    elif s1=='Florida Marlins':
        return 'MIA'
    elif 'Chicago White Sox' in s1:
        return 'CHW'
    elif "Angels" in s1:
        return "LAA"
    elif "Tampa Bay" in s1:
        return "TBR"

    for code in codes:
        for item in L:
            if item==code:
                return code

    return (s1,'No Match')

# 1. Ensure your 'codes' list contains the 30 active abbreviations
# (Based on our previous list: LAD, PHI, BAL, CIN, MIL, etc.)
codes = rosters['Team'].unique()

# 2. Apply the match function to create the 'Team' column in all dataframes
# We use a lambda to ensure we get a string back even on 'No Match'
for df in [career_hitting, career_fielding, career_pitching]:
    df['Team'] = df['team_name'].apply(lambda x: match(x, codes))
    
    # Cleaning up the 'No Match' tuple if it occurs
    df['Team'] = df['Team'].apply(lambda x: x if isinstance(x, str) else "No Match")

# 3. Quick verification check
print(f"Unique Teams in Hitting: {career_hitting['Team'].nunique()}")
print(career_hitting[['team_name', 'Team']].head())

for name, df in zip(['Hitting', 'Fielding', 'Pitching'], [career_hitting, career_fielding, career_pitching]):
    no_match_count = (df['Team'] == "No Match").sum()
    total_rows = len(df)
    percent = (no_match_count / total_rows) * 100
    print(f"{name} - No Match: {no_match_count} ({percent:.1f}%)")

roster_codes = set(rosters['Team'].unique())

for name, df in zip(['Hitting', 'Fielding', 'Pitching'], [career_hitting, career_fielding, career_pitching]):
    no_match = (df['Team'] == 'No Match').sum()
    unmatched_codes = set(df[df['Team'] != 'No Match']['Team'].unique()) - roster_codes
    missing_from_career = roster_codes - set(df['Team'].unique())
    
    print(f"\n── {name} ──")
    print(f"  Total rows:            {len(df)}")
    print(f"  No Match:              {no_match} ({no_match/len(df)*100:.1f}%)")
    print(f"  Codes not in rosters:  {unmatched_codes if unmatched_codes else '✓ None'}")
    print(f"  Roster teams missing:  {missing_from_career if missing_from_career else '✓ All 30 present'}")


Unique Teams in Hitting: 30
                team_name Team
3     Los Angeles Dodgers  LAD
9   Philadelphia Phillies  PHI
12      Baltimore Orioles  BAL
13        Cincinnati Reds  CIN
14  Philadelphia Phillies  PHI
Hitting - No Match: 0 (0.0%)
Fielding - No Match: 0 (0.0%)
Pitching - No Match: 0 (0.0%)

── Hitting ──
  Total rows:            13338
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}

── Fielding ──
  Total rows:            31317
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}

── Pitching ──
  Total rows:            7768
  No Match:              0 (0.0%)
  Codes not in rosters:  ✓ None
  Roster teams missing:  {nan}


## Two-Way Player & Position Filtering
Ensure players are assigned to the correct career stat datasets based on position.
Pitchers are removed from the hitting set, and the pitching set is restricted to
pitchers and two-way players only. Ohtani is used as a spot-check to confirm
two-way players appear correctly in both datasets.

In [105]:
from IPython.display import display

# ── Hitting ──
print("=== HITTING ===")
print("Unique Position Abbreviations:", career_hitting['position_abbr'].unique())
print("Unique Position Names:        ", career_hitting['position_name'].unique())
display(career_hitting[career_hitting['first_name'] == 'Shohei'])

career_hitting = career_hitting[career_hitting['position_name'] != 'Pitcher']
print(f"Rows remaining after removing Pitchers: {len(career_hitting)}\n")

# ── Pitching ──
print("=== PITCHING ===")
print("Unique Position Abbreviations:", career_pitching['position_abbr'].unique())
print("Unique Position Names:        ", career_pitching['position_name'].unique())
display(career_pitching[career_pitching['first_name'] == 'Shohei'])

career_pitching = career_pitching[career_pitching['position_name'].isin(['Pitcher', 'Two-Way Player'])]
print(f"Rows remaining after filtering for Pitcher/Two-Way: {len(career_pitching)}")

=== HITTING ===
Unique Position Abbreviations: ['P' 'PH' 'C' 'RF' 'X' '3B' 'LF' '2B' 'SS' '1B' 'CF' 'DH' 'PR']
Unique Position Names:         ['Pitcher' 'Pinch Hitter' 'Catcher' 'Outfielder' 'Unknown' 'Third Base'
 'Second Base' 'Shortstop' 'First Base' 'Designated Hitter' 'Pinch Runner']


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
2400,660271,Shohei Ohtani,Shohei,Ohtani,10,Designated Hitter,DH,119.0,Los Angeles Dodgers,104.0,...,49.0,20.0,.325,800.0,796.0,17041.0,1420.0,1.01,13.0,LAD


Rows remaining after removing Pitchers: 7194

=== PITCHING ===
Unique Position Abbreviations: ['CF' 'SS' '1B' 'P' 'C' 'DH' '2B' '3B' 'RF' 'LF' 'TWP' 'OF']
Unique Position Names:         ['Outfielder' 'Shortstop' 'First Base' 'Pitcher' 'Catcher'
 'Designated Hitter' 'Second Base' 'Third Base' 'Two-Way Player'
 'Outfield']


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
1325,660271,Shohei Ohtani,Shohei,Ohtani,Y,Two-Way Player,TWP,119.0,Los Angeles Dodgers,104.0,...,3.1,6.61,3.13,0.95,0.0,0.0,2.0,2.0,12.0,LAD


Rows remaining after filtering for Pitcher/Two-Way: 7238


## Restricted List Players — Team Assignment & Removal
Manually assign teams for players with null Team values, then drop all restricted
list players since they are ineligible to play and carry null projections.

In [107]:
# All four are on restricted lists — manually assign their teams
team_map = {
    'Luis Ortiz':        'CLE',   # Cleveland Guardians — on restricted list, pitch-rigging case
    'Emmanuel Clase':    'CLE',   # Cleveland Guardians — on restricted list, pitch-rigging case
    'Michael Lorenzen':  'COL',   # Colorado Rockies — signed Jan 2026
    'Wander Franco':     'TBR',   # Tampa Bay Rays — on restricted list, legal issues in DR
}

for name, team in team_map.items():
    rosters.loc[rosters['Name'] == name, 'Team'] = team
print("Restricted players being dropped:")
print(rosters[rosters['Status'] == 'RESTRICTED'][['Name', 'Team', 'Pos', 'ProjectedOpeningDayRole']])

rosters = rosters[rosters['Status'] != 'RESTRICTED']
print(f"\nRoster size after dropping restricted players: {len(rosters)}")

print('Null Team Count in Rosters: ', rosters['Team'].isnull().sum())

Restricted players being dropped:
                  Name Team Pos ProjectedOpeningDayRole
204   Jurickson Profar  ATL  OF         Restricted List
591         Luis Ortiz  CLE  SP         Restricted List
592     Emmanuel Clase  CLE  RP         Restricted List
1839     Wander Franco  TBR  SS         Restricted List

Roster size after dropping restricted players: 2057
Null Team Count in Rosters:  0


In [108]:
rosters[rosters['Team']=='ATH']
print(f"Rosters unique teams:          {rosters['Team'].nunique()}")
print(f"Career hitting unique teams:   {career_hitting['team_name'].nunique()}")
print(f"Career fielding unique teams:  {career_fielding['team_name'].nunique()}")
print(f"Career pitching unique teams:  {career_pitching['team_name'].nunique()}")

# Also check what AL West teams are in rosters
print("\nAL West teams in rosters:")
print(rosters[rosters['Team'].isin(['HOU','SEA','TEX','LAA','OAK','ATH'])]['Team'].unique())
print('Fielding')
print(career_fielding['team_name'].unique())
print(sorted(career_fielding['Team'].unique()))
print()
print(career_hitting['team_name'].unique())
print(sorted(career_hitting['Team'].unique()))
print()

print(sorted(rosters['Team'].unique()))

Rosters unique teams:          30
Career hitting unique teams:   30
Career fielding unique teams:  30
Career pitching unique teams:  30

AL West teams in rosters:
['ATH' 'HOU' 'LAA' 'SEA' 'TEX']
Fielding
['St. Louis Cardinals' 'Boston Red Sox' 'Pittsburgh Pirates'
 'San Francisco Giants' 'Los Angeles Dodgers' 'Philadelphia Phillies'
 'Kansas City Royals' 'Chicago White Sox' 'Cincinnati Reds'
 'Detroit Tigers' 'New York Mets' 'Chicago Cubs' 'Houston Astros'
 'Baltimore Orioles' 'New York Yankees' 'San Diego Padres' 'Texas Rangers'
 'Minnesota Twins' 'Los Angeles Angels' 'Milwaukee Brewers'
 'Seattle Mariners' 'Arizona Diamondbacks' 'Toronto Blue Jays'
 'Atlanta Braves' 'Colorado Rockies' 'Tampa Bay Rays' 'Miami Marlins'
 'Washington Nationals' 'Athletics' 'Cleveland Guardians']
['ARI', 'ATH', 'ATL', 'BAL', 'BOS', 'CHC', 'CHW', 'CIN', 'CLE', 'COL', 'DET', 'HOU', 'KCR', 'LAA', 'LAD', 'MIA', 'MIL', 'MIN', 'NYM', 'NYY', 'PHI', 'PIT', 'SDP', 'SEA', 'SFG', 'STL', 'TBR', 'TEX', 'TOR', 'WSN']



## **Duplicate Name Audit**
Identifies active roster players whose names map to multiple `player_id` entries
in the historical data — indicating a name conflict between two different players.
These cases require manual vetting to ensure career stats are merged onto the
correct player.

In [110]:
def get_at_risk_names(roster_df, history_df, roster_name_col='Name', history_name_col='full_name'):
    """
    Identifies names in the active roster that have multiple 
    player_id entries in the historical dataset.
    """
    # 1. Identify all names in history associated with more than one player_id
    historical_id_counts = history_df.groupby(history_name_col)['player_id'].nunique()
    historical_duplicate_names = historical_id_counts[historical_id_counts > 1].index
    
    # 2. Find which roster names are in that historical duplicate list
    at_risk_mask = roster_df[roster_name_col].isin(historical_duplicate_names)
    at_risk_names = roster_df.loc[at_risk_mask, roster_name_col].unique().tolist()
    
    # Printing summary for visibility
    print(f"--- Conflict Audit ---")
    print(f"Total historical names with multiple IDs: {len(historical_duplicate_names)}")
    print(f"Active roster names flagged for manual vetting: {len(at_risk_names)}")
    
    return at_risk_names
all_positions = ['SS', '2B', 'RF', '3B', 'CF', 'C', '1B', 'DH', 'LF', 'INF/OF',
                 'SP', 'OF', 'RP', 'SP/RP', 'INF', '1B/3B', '3B/1B', '1B/OF',
                 'C/1B', 'OF/1B', 'OF/INF', 'C/INF', '1B/C', 'SS/2B', 'UTL',
                 'RP/SP', 'SS/2B/3B', 'CF/RF/LF', 'C/OF', '1B/LF', '2B/3B/1B']

pitching_positions = [pos for pos in all_positions if 'P' in pos]
print(f"Pitching positions: {pitching_positions}")
pitching_rosters=rosters[rosters['Pos'].isin(pitching_positions)]
hitting_rosters=rosters[~rosters['Pos'].isin(pitching_positions)]

# --- Apply to Hitting ---
print("\nHITTING AUDIT:")
hitting_at_risk = get_at_risk_names(hitting_rosters, career_hitting)
print(hitting_at_risk)
# --- Apply to Pitching ---
print("\nPITCHING AUDIT:")
pitching_at_risk = get_at_risk_names(pitching_rosters, career_pitching)
print(pitching_at_risk)
# --- Apply to Fielding ---
# Note: Since fielding usually covers both pitchers and hitters, 
# you can check it against the full rosters.
print("\nFIELDING AUDIT:")
fielding_at_risk = get_at_risk_names(rosters, career_fielding)
print(fielding_at_risk)


Pitching positions: ['SP', 'RP', 'SP/RP', 'RP/SP']

HITTING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 64
Active roster names flagged for manual vetting: 5
['Jacob Wilson', 'Max Muncy', 'Nick Allen', 'Josh Bell', 'Luis Matos']

PITCHING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 66
Active roster names flagged for manual vetting: 6
['Eduardo Rodriguez', 'Juan Morillo', 'Danny Young', 'Javy Guerra', 'Logan Allen', 'Carlos Hernández']

FIELDING AUDIT:
--- Conflict Audit ---
Total historical names with multiple IDs: 196
Active roster names flagged for manual vetting: 13
['Eduardo Rodriguez', 'Ryan Thompson', 'Max Muncy', 'Danny Young', 'Javy Guerra', 'José Ramírez', 'Carlos Hernández', 'Nick Allen', 'Will Smith', 'Eury Pérez', 'Josh Bell', 'Luis Matos', 'Jose Herrera']


## **Roster Splitting, Duplicate Removal & Career Stat Merging**
Positions are split into pitching (`SP`, `RP`, `SP/RP`, `RP/SP`) and hitting sets.
Flagged duplicate names are isolated and removed from both rosters and career stat
dataframes before merging. Hitting and fielding stats are collapsed and joined onto
the hitting roster, while pitching stats are merged onto the pitching roster.
Pitching columns are prefixed with `P_` to avoid conflicts during later combination.

In [112]:
# ── Position Splitting ──────────────────────────────────────────────────────
all_positions = ['SS', '2B', 'RF', '3B', 'CF', 'C', '1B', 'DH', 'LF', 'INF/OF',
                 'SP', 'OF', 'RP', 'SP/RP', 'INF', '1B/3B', '3B/1B', '1B/OF',
                 'C/1B', 'OF/1B', 'OF/INF', 'C/INF', '1B/C', 'SS/2B', 'UTL',
                 'RP/SP', 'SS/2B/3B', 'CF/RF/LF', 'C/OF', '1B/LF', '2B/3B/1B']

pitching_positions = [pos for pos in all_positions if 'P' in pos]
print(f"Pitching positions: {pitching_positions}")

pitching_rosters = rosters[rosters['Pos'].isin(pitching_positions)]
hitting_rosters  = rosters[~rosters['Pos'].isin(pitching_positions)]
print(f"Hitting roster size:  {len(hitting_rosters)}")
print(f"Pitching roster size: {len(pitching_rosters)}")

# ── Remove Duplicate-Name Players from Rosters and Career Stats ─────────────
# Isolate flagged rows for inspection
hitting_rosters_dup_names  = hitting_rosters[hitting_rosters['Name'].isin(hitting_at_risk)]
pitching_rosters_dup_names = pitching_rosters[pitching_rosters['Name'].isin(pitching_at_risk)]

# Clean rosters — remove ambiguous names
clean_hitting_rosters  = hitting_rosters[~hitting_rosters['Name'].isin(hitting_at_risk)]
clean_pitching_rosters = pitching_rosters[~pitching_rosters['Name'].isin(pitching_at_risk)]

# Clean career stats — remove ambiguous names from all three datasets
all_risky_names        = hitting_at_risk + pitching_at_risk
clean_career_hitting   = career_hitting[~career_hitting['full_name'].isin(all_risky_names)]
clean_career_fielding  = career_fielding[~career_fielding['full_name'].isin(all_risky_names)]
clean_career_pitching  = career_pitching[~career_pitching['full_name'].isin(all_risky_names)]

print(f"\nPlayers removed due to name conflicts: {len(all_risky_names)}")
print(f"  Hitting:  {len(hitting_rosters)} → {len(clean_hitting_rosters)} roster rows")
print(f"  Pitching: {len(pitching_rosters)} → {len(clean_pitching_rosters)} roster rows")

# ── Merge Hitting + Career Hitting ──────────────────────────────────────────
merged_hitting = pd.merge(clean_hitting_rosters, clean_career_hitting.drop(columns=['Team']),
                          left_on='Name', right_on='full_name', how='inner')
print(f"\nHitting merge: {len(clean_hitting_rosters)} roster → {len(merged_hitting)} after inner join")

# ── Collapse Fielding Stats (one row per player) ─────────────────────────────
# Fielding can have multiple rows per player (one per position played)
# We collapse to one row per player using gamesPlayed as a tiebreaker
fielding_columns = [
    'first_name', 'last_name', 'full_name', 'player_id', 'gamesPlayed',
    'gamesStarted', 'assists', 'putOuts', 'errors', 'chances', 'fielding',
    'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'games',
    'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall',
    'caughtStealing', 'stolenBases', 'stolenBasePercentage',
    'caughtStealingPercentage', 'catcherERA', 'catchersInterference',
    'wildPitches', 'pickoffs', 'position'
]

# These columns already exist in the hitting merge — drop to avoid _x/_y conflicts
duplicate_columns = [
    'catchersInterference', 'caughtStealing', 'caughtStealingPercentage',
    'gamesPlayed', 'stolenBasePercentage', 'stolenBases'
]
exclude_cols = ['position_code', 'position_name', 'position_abbr', 'position'] + duplicate_columns

fielding_collapsed       = clean_career_fielding.groupby(['first_name', 'last_name', 'gamesPlayed'], as_index=False).first()
fielding_columns_reduced = [col for col in fielding_columns if col not in exclude_cols]
fielding_reduced         = fielding_collapsed[fielding_columns_reduced]

# ── Merge Offense (Hitting + Fielding) ───────────────────────────────────────
merged_offense = pd.merge(merged_hitting, fielding_reduced, on='full_name', how='inner')
print(f"Offense merge (hitting + fielding): {len(merged_offense)} rows")

# ── Merge Pitching + Career Pitching ────────────────────────────────────────
merged_pitching = pd.merge(pitching_rosters, clean_career_pitching.drop(columns=['Team']),
                           left_on='Name', right_on='full_name', how='inner')
print(f"Pitching merge: {len(pitching_rosters)} roster → {len(merged_pitching)} after inner join")

# ── Prefix Pitching Columns with P_ ─────────────────────────────────────────
# Roster columns keep their names; all stat columns get a P_ prefix
# to avoid conflicts when offense and pitching are later combined
def reframe_columns(col):
    roster_columns = [
        'Team', 'Team_x', 'Name', 'Pos', 'Status', 'ProjectedOpeningDayRole',
        'Age', 'ServiceTime', 'Is40Man', 'Options', 'Proj_PA', 'Proj_IP',
        'Proj_PT', 'first_name', 'last_name'
    ]
    if col in roster_columns:
        return 'Team' if col in ['Team_x', 'P_Team_x', 'P_P_Team_x'] else col
    return f"P_{col}"

merged_pitching.columns = [reframe_columns(col) for col in merged_pitching.columns]

# ── Verification ─────────────────────────────────────────────────────────────
print(f"\n── Final Shapes ──")
print(f"  merged_offense:  {merged_offense.shape}")
print(f"  merged_pitching: {merged_pitching.shape}")
print(f"\n── Duplicate name rows set aside for manual review ──")
print(f"  Hitting:  {len(hitting_rosters_dup_names)} players")
print(f"  Pitching: {len(pitching_rosters_dup_names)} players")
display(hitting_rosters_dup_names[['Name', 'Pos', 'Team']]) if len(hitting_rosters_dup_names) > 0 else print("  None")
display(pitching_rosters_dup_names[['Name', 'Pos', 'Team']]) if len(pitching_rosters_dup_names) > 0 else print("  None")

Pitching positions: ['SP', 'RP', 'SP/RP', 'RP/SP']
Hitting roster size:  961
Pitching roster size: 1096

Players removed due to name conflicts: 11
  Hitting:  961 → 955 roster rows
  Pitching: 1096 → 1090 roster rows

Hitting merge: 955 roster → 653 after inner join
Offense merge (hitting + fielding): 788 rows
Pitching merge: 1096 roster → 809 after inner join

── Final Shapes ──
  merged_offense:  (788, 79)
  merged_pitching: (809, 87)

── Duplicate name rows set aside for manual review ──
  Hitting:  6 players
  Pitching: 6 players


,Name,Pos,Team
79,Jacob Wilson,SS,ATH
84,Max Muncy,3B,ATH
758,Nick Allen,INF,HOU
956,Max Muncy,3B,LAD
1157,Josh Bell,1B,MIN
1668,Luis Matos,OF,SFG


,Name,Pos,Team
14,Eduardo Rodriguez,SP,ARI
43,Juan Morillo,RP,ARI
178,Danny Young,RP,ATL
212,Javy Guerra,RP,ATL
559,Logan Allen,SP,CLE
585,Carlos Hernández,RP,CLE


## Manual Resolution of Duplicate Hitting Names
Manually inspect and resolve players whose names matched multiple `player_id` entries.
Incorrect historical player records are dropped by `player_id`, and name conflicts
(e.g. two players named Max Muncy) are disambiguated by renaming one entry before merging.
Resolved duplicate rows are then concatenated back into the main `merged_offense_clean` dataset.

In [114]:
hitting_names=hitting_rosters_dup_names['Name'].values

for name in hitting_names:
    display(career_hitting[career_hitting['full_name']==name].sort_values(by='full_name'))
    print()
career_stats_dups_hitting=career_hitting[career_hitting['full_name'].isin(hitting_names)].sort_values(by='full_name').reset_index(drop=True)
print('Career Stats Hitting')
display(career_stats_dups_hitting)
idx_to_drop=[458679,607111,275928,110139]
# Create a new dataframe without those IDs
career_stats_dups_hitting = career_stats_dups_hitting[~career_stats_dups_hitting['player_id'].isin(idx_to_drop)]
print('Career Stats After Drop')
career_stats_dups_hitting.at[6,'full_name']='Max Muncy Dodger'


display(career_stats_dups_hitting)

hitting_rosters_dups=hitting_rosters[hitting_rosters['Name'].isin(hitting_names)].sort_values(by='Name').reset_index(drop=True)
hitting_rosters_dups.at[4,'Name']='Max Muncy Dodger'
merged_hitting_dups=pd.merge(hitting_rosters_dups,career_stats_dups_hitting.drop(columns=['Team']), left_on='Name', right_on='full_name')
print('Joined with Rosters')
display(merged_hitting_dups)

career_stats_dups_fielding=career_fielding[career_fielding['full_name'].isin(hitting_names)].sort_values(by='full_name').reset_index(drop=True)
print('Career Stats fielding')
display(career_stats_dups_fielding)

career_stats_dups_fielding=career_stats_dups_fielding.groupby('player_id').first().sort_values(by='full_name').reset_index()
career_stats_dups_fielding.at[6,'full_name']='Max Muncy Dodger'

career_stats_dups_fielding=career_stats_dups_fielding[~career_stats_dups_fielding['player_id'].isin(idx_to_drop)]
print('Feilding after drop')
display(career_stats_dups_fielding)
career_stats_dups_fielding=career_stats_dups_fielding[fielding_columns_reduced]
merged_offense_dups=pd.merge(merged_hitting_dups,career_stats_dups_fielding, on='full_name')
merged_offense_dups

merged_offense_clean=pd.concat([merged_offense,merged_offense_dups])

pitching_rosters_dup_names
print('all code executed')

,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
1406,805779,Jacob Wilson,Jacob,Wilson,6,Shortstop,SS,133.0,Athletics,103.0,...,20.0,2.0,.311,216.0,144.0,2084.0,211.0,1.50,0.0,ATH
14150,607111,Jacob Wilson,Jacob,Wilson,5,Third Base,3B,117.0,Houston Astros,103.0,...,1.0,0.0,.176,9.0,5.0,85.0,9.0,1.80,0.0,HOU


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8675,571970,Max Muncy,Max,Muncy,5,Third Base,3B,119.0,Los Angeles Dodgers,104.0,...,36.0,33.0,.252,689.0,913.0,17004.0,1547.0,0.75,0.0,LAD
10155,691777,Max Muncy,Max,Muncy,5,Third Base,3B,133.0,Athletics,103.0,...,1.0,1.0,.269,51.0,44.0,824.0,79.0,1.16,0.0,ATH


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8363,110139,Nick Allen,Nick,Allen,2,Catcher,C,113.0,Cincinnati Reds,104.0,...,NaN,NaN,.272,NaN,NaN,NaN,NaN,NaN,NaN,CIN
10184,669397,Nick Allen,Nick,Allen,6,Shortstop,SS,144.0,Atlanta Braves,104.0,...,21.0,6.0,.263,345.0,291.0,4260.0,413.0,1.19,0.0,ATL


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
8675,571970,Max Muncy,Max,Muncy,5,Third Base,3B,119.0,Los Angeles Dodgers,104.0,...,36.0,33.0,.252,689.0,913.0,17004.0,1547.0,0.75,0.0,LAD
10155,691777,Max Muncy,Max,Muncy,5,Third Base,3B,133.0,Athletics,103.0,...,1.0,1.0,.269,51.0,44.0,824.0,79.0,1.16,0.0,ATH


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,groundIntoDoublePlay,sacFlies,babip,groundOuts,airOuts,numberOfPitches,leftOnBase,groundOutsToAirouts,catchersInterference,Team
5106,605137,Josh Bell,Josh,Bell,3,First Base,1B,120.0,Washington Nationals,104.0,...,139.0,41.0,.283,1381.0,1048.0,20288.0,2039.0,1.32,10.0,WSN
11632,458679,Josh Bell,Josh,Bell,5,Third Base,3B,110.0,Baltimore Orioles,103.0,...,8.0,0.0,.278,74.0,53.0,1058.0,131.0,1.40,0.0,BAL


...

Career Stats fielding


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,catcherERA,catchersInterference,wildPitches,pickoffs,position,Team
0,805779,Jacob Wilson,Jacob,Wilson,6,Shortstop,SS,133.0,Athletics,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '6', 'name': 'Shortstop', 'type': 'In...",ATH
1,805779,Jacob Wilson,Jacob,Wilson,10,Designated Hitter,DH,133.0,Athletics,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '6', 'name': 'Shortstop', 'type': 'In...",ATH
2,605137,Josh Bell,Josh,Bell,10,Designated Hitter,DH,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
3,605137,Josh Bell,Josh,Bell,3,First Base,1B,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
4,458679,Josh Bell,Josh,Bell,10,Designated Hitter,DH,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
5,458679,Josh Bell,Josh,Bell,9,Outfielder,RF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
6,458679,Josh Bell,Josh,Bell,7,Outfielder,LF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
7,458679,Josh Bell,Josh,Bell,5,Third Base,3B,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL
8,605137,Josh Bell,Josh,Bell,9,Outfielder,RF,120.0,Washington Nationals,104.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
9,275928,Luis Matos,Luis,Matos,8,Outfielder,CF,110.0,Baltimore Orioles,103.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAL


Feilding after drop


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,catcherERA,catchersInterference,wildPitches,pickoffs,position,Team
0,805779,Jacob Wilson,Jacob,Wilson,6,Shortstop,SS,133.0,Athletics,103.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '6', 'name': 'Shortstop', 'type': 'In...",ATH
2,605137,Josh Bell,Josh,Bell,10,Designated Hitter,DH,120.0,Washington Nationals,104.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",WSN
4,682641,Luis Matos,Luis,Matos,9,Outfielder,RF,137.0,San Francisco Giants,104.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '7', 'name': 'Outfielder', 'type': 'O...",SFG
5,571970,Max Muncy,Max,Muncy,10,Designated Hitter,DH,119.0,Los Angeles Dodgers,104.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '3', 'name': 'First Base', 'type': 'I...",LAD
6,691777,Max Muncy Dodger,Max,Muncy,5,Third Base,3B,133.0,Athletics,103.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '4', 'name': 'Second Base', 'type': '...",ATH
8,669397,Nick Allen,Nick,Allen,4,Second Base,2B,144.0,Atlanta Braves,104.0,...,NaN,NaN,None,None,None,NaN,NaN,NaN,"{'code': '4', 'name': 'Second Base', 'type': '...",ATL


all code executed


## Manual Resolution of Duplicate Pitching Names
Inspect pitchers whose names matched multiple historical records and resolve by
selecting the entry with the highest `player_id` (most recent player). Resolved
rows are prefixed with `P_` and concatenated back into `merged_pitching_clean`.

In [116]:
# ── Inspect duplicate pitching names ────────────────────────────────────────
print('Current Roster Pitchers with Historical Duplicates')
display(pitching_rosters_dup_names.sort_values(by='Name'))

# Pull all career pitching rows for flagged names
career_pitching_dups = career_pitching[career_pitching['full_name'].isin(all_risky_names)]
print('All Career Pitching Rows for Flagged Names')
display(career_pitching_dups.sort_values(by='full_name'))

# ── Resolve duplicates by keeping the highest player_id (most recent) ────────
career_pitching_dups_reduced = career_pitching_dups.groupby('full_name').apply(
    lambda x: x.loc[x['player_id'].idxmax()]
).reset_index(drop=True)
print('After Removing Inactive Players (highest player_id kept)')
display(career_pitching_dups_reduced.sort_values(by='full_name'))

# ── Merge resolved duplicates onto their roster rows ────────────────────────
merged_pitching_dups = pd.merge(
    pitching_rosters_dup_names, career_pitching_dups_reduced.drop(columns=['Team']),
    left_on='Name', right_on='full_name'
)
print('Duplicate Pitchers Merged with Roster')
display(merged_pitching_dups)

# ── Apply P_ prefix and concatenate back into main pitching dataset ──────────
merged_pitching_dups.columns = [reframe_columns(col) for col in merged_pitching_dups.columns]
merged_pitching_clean = pd.concat([merged_pitching, merged_pitching_dups])
print(f'merged_pitching_clean: {merged_pitching_clean.shape}')

Current Roster Pitchers with Historical Duplicates


,Team,Name,Pos,Status,ProjectedOpeningDayRole,Age,ServiceTime,Is40Man,Options,Proj_PA,Proj_IP,Proj_PT,first_name,last_name
585,CLE,Carlos Hernández,RP,NRI,Bullpen Candidate,29.0,4.075,N,0,NaN,18.0,18.0,Carlos,Hernández
178,ATL,Danny Young,RP,Inj,Projected Injured List,31.8,1.160,Y,0,NaN,20.0,20.0,Danny,Young
14,ARI,Eduardo Rodriguez,SP,ACTIVE,Starting Rotation,32.9,10.070,Y,NaN,NaN,132.0,132.0,Eduardo,Rodriguez
212,ATL,Javy Guerra,RP,NRI,Reassigned,30.5,3.003,N,0,NaN,NaN,NaN,Javy,Guerra
43,ARI,Juan Morillo,RP,MiLB (40),Bullpen Candidate,27.0,0.111,Y,2,NaN,30.0,30.0,Juan,Morillo
559,CLE,Logan Allen,SP,ACTIVE,Starting Rotation,27.5,2.093,Y,1,NaN,121.0,121.0,Logan,Allen


All Career Pitching Rows for Flagged Names


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
6192,400056,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,117.0,Houston Astros,104.0,...,4.8,9.12,4.69,1.21,1.0,0.0,0.0,7.0,3.0,HOU
7715,672578,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0,CLE
4199,664849,Danny Young,Danny,Young,1,Pitcher,P,121.0,New York Mets,104.0,...,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0,NYM
11623,277415,Danny Young,Danny,Young,1,Pitcher,P,112.0,Chicago Cubs,104.0,...,18.00,15.00,21.00,3.00,0.0,0.0,0.0,0.0,0.0,CHC
3775,121350,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,158.0,Milwaukee Brewers,103.0,...,3.96,8.35,4.27,0.8,242.0,77.0,1.0,43.0,30.0,MIL
4900,593958,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0,ARI
4107,457915,Javy Guerra,Javy,Guerra,1,Pitcher,P,119.0,Los Angeles Dodgers,104.0,...,3.6,9.13,4.28,0.86,127.0,53.0,1.0,14.0,11.0,LAD
9482,642770,Javy Guerra,Javy,Guerra,1,Pitcher,P,139.0,Tampa Bay Rays,103.0,...,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0,TBR
4924,666661,Juan Morillo,Juan,Morillo,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0,ARI
11287,434623,Juan Morillo,Juan,Morillo,1,Pitcher,P,115.0,Colorado Rockies,104.0,...,5.91,12.66,13.50,4.22,0.0,0.0,0.0,1.0,0.0,COL


After Removing Inactive Players (highest player_id kept)


,player_id,full_name,first_name,last_name,position_code,position_name,position_abbr,team_id,team_name,league_id,...,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,Team
0,672578,Carlos Hernández,Carlos,Hernández,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0,CLE
1,664849,Danny Young,Danny,Young,1,Pitcher,P,121.0,New York Mets,104.0,...,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0,NYM
2,593958,Eduardo Rodriguez,Eduardo,Rodriguez,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0,ARI
3,642770,Javy Guerra,Javy,Guerra,1,Pitcher,P,139.0,Tampa Bay Rays,103.0,...,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0,TBR
4,666661,Juan Morillo,Juan,Morillo,1,Pitcher,P,109.0,Arizona Diamondbacks,104.0,...,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0,ARI
5,671106,Logan Allen,Logan,Allen,1,Pitcher,P,114.0,Cleveland Guardians,103.0,...,3.58,9.42,4.79,1.33,0.0,0.0,2.0,5.0,10.0,CLE


Duplicate Pitchers Merged with Roster


,Team,Name,Pos,Status,ProjectedOpeningDayRole,Age,ServiceTime,Is40Man,Options,Proj_PA,...,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies
0,ARI,Eduardo Rodriguez,SP,ACTIVE,Starting Rotation,32.9,10.070,Y,NaN,NaN,...,8.95,3.14,8.83,4.42,1.15,2.0,0.0,2.0,19.0,34.0
1,ARI,Juan Morillo,RP,MiLB (40),Bullpen Candidate,27.0,0.111,Y,2,NaN,...,9.44,5.24,9.96,4.98,0.52,22.0,10.0,0.0,2.0,1.0
2,ATL,Danny Young,RP,Inj,Projected Injured List,31.8,1.160,Y,0,NaN,...,11.57,3.71,8.31,4.6,0.59,32.0,7.0,0.0,3.0,0.0
3,ATL,Javy Guerra,RP,NRI,Reassigned,30.5,3.003,N,0,NaN,...,6.43,6.14,9.86,6.71,1.29,18.0,7.0,0.0,1.0,3.0
4,CLE,Logan Allen,SP,ACTIVE,Starting Rotation,27.5,2.093,Y,1,NaN,...,7.59,3.58,9.42,4.79,1.33,0.0,0.0,2.0,5.0,10.0
5,CLE,Carlos Hernández,RP,NRI,Bullpen Candidate,29.0,4.075,N,0,NaN,...,7.96,4.41,8.89,5.47,0.99,67.0,25.0,0.0,2.0,15.0


merged_pitching_clean: (815, 87)


## **Column Cleanup — Drop `_y` Duplicates and Strip `_x` Suffixes**
Remove duplicate `_y` columns introduced by merges and clean up residual `_x` suffixes.

In [118]:
import re
merged_offense_clean=merged_offense_clean[[col for col in merged_offense_clean.columns if '_y' not in col]]
merged_pitching_clean=merged_pitching_clean[[col for col in merged_pitching_clean.columns if '_y' not in col]]
for col in merged_offense_clean.columns:
    if 'x' in col:
        print(col)
print()
for col in merged_pitching_clean.columns:
    if 'x' in col:
        print(col)
merged_offense_clean.columns=[ re.sub('_x',"",col) for col in merged_offense_clean.columns]
merged_pitching_clean.columns=[ re.sub('_x',"",col) for col in merged_pitching_clean.columns]

first_name_x
last_name_x
player_id_x

P_first_name_x
P_last_name_x


## Null Handling & Player Weight Calculation
Catcher-specific stats (`passedBall`, `wildPitches`, `pickoffs`, `catcherERA`) are filled
with zero for non-catchers. Projection columns (`Proj_PA`, `Proj_IP`, `Proj_PT`) are
zero-filled for players without projections. Player weights are then calculated as each
player's share of their team's total projected plate appearances (offense) or innings
pitched (pitching), so weights sum to 1.0 per team.

In [120]:
print('Length of Merged Offense: ', len(merged_offense_clean))
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")

catcher_columns=["passedBall",
"catcherERA",
"wildPitches",
"pickoffs"]
catcher=merged_offense_clean[catcher_columns+['Name','position_name']]


# .any(axis=1) collapses the 4 columns into a single True/False Series
catcher_filtered = catcher[catcher[catcher_columns].notnull().any(axis=1)]
display(catcher_filtered)
# Only these columns get filled; others stay null
merged_offense_clean.fillna({
    "passedBall": 0,
    "wildPitches": 0,
    "pickoffs": 0,
    "catcherERA": 0  # Or maybe fill with the league average?
}, inplace=True)
print('Catcher/First Basemen Stats filled with zero if null')
print()
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")
# Filter for rows where Proj_PA is not null
null_plate_appearances = merged_offense_clean[merged_offense_clean['Proj_PA'].isnull()]

# Optional: Sort by projected plate appearances to see your heavy hitters at the top
null_plate_appearances  = null_plate_appearances .sort_values(by='Proj_PA', ascending=False)
display(null_plate_appearances [['Name','Team','Proj_PA','ServiceTime','Is40Man','Status']])

# Apply zero-fill to the Offense projections
offense_proj_cols = ['Proj_PT', 'Proj_PA', 'Proj_IP']

merged_offense_clean[offense_proj_cols] = merged_offense_clean[offense_proj_cols].fillna(0)

# Verify the fix for Offense
print("Offense Nulls Remaining:")
print(merged_offense_clean[offense_proj_cols].isnull().sum())




# Calculate total projected PA per team
team_pa_totals = merged_offense_clean.groupby('Team')['Proj_PA'].sum()

# Weight = Player PA / Team Total PA
# We use .get() or a mapping to ensure the 'Team' index matches correctly
merged_offense_clean['Player_Weight'] = merged_offense_clean.apply(
    lambda x: x['Proj_PA'] / team_pa_totals[x['Team']] if team_pa_totals[x['Team']] > 0 else 0, 
    axis=1
)
merged_pitching_clean['Proj_IP'] = merged_pitching_clean['Proj_IP'].fillna(0)
# 1. Calculate the team total IP and broadcast it to every row
team_ip_totals = merged_pitching_clean.groupby('Team')['Proj_IP'].transform('sum')

# 2. Calculate the weight (Player IP / Team Total IP)
# We use a small epsilon (1e-9) to prevent division by zero for teams with no IP
merged_pitching_clean['P_Player_Weight'] = merged_pitching_clean['Proj_IP'] / (team_ip_totals + 1e-9)

# 3. Clean up any cases where both were 0 (result would be 0.0)
merged_pitching_clean['P_Player_Weight'] = merged_pitching_clean['P_Player_Weight'].fillna(0)

# Apply the fill to both cleaned dataframes
merged_offense_clean['ServiceTime'] = merged_offense_clean['ServiceTime'].fillna(0)
merged_pitching_clean['ServiceTime'] = merged_pitching_clean['ServiceTime'].fillna(0)

# Verify: Sum of weights for the 'LAD' or 'NYY' should be approx 1.0
print('Verfify Sums add to 1')
print('Pitching')
print(merged_pitching_clean.groupby('Team')['P_Player_Weight'].sum().head())
print('Offense')
print(merged_offense_clean.groupby('Team')['Player_Weight'].sum().head())

def check_weights(df, weight_col, label):
    # Calculate sums per team
    team_sums = df.groupby('Team')[weight_col].sum()
    
    # Define a tiny buffer (epsilon)
    epsilon = 1e-6
    
    # Find teams that are NOT approximately 1.0
    # Note: We exclude teams with 0.0 sum (likely teams with no projections)
    invalid_teams = team_sums[(team_sums > 0) & ((team_sums < 1 - epsilon) | (team_sums > 1 + epsilon))]
    
    print(f"--- {label} Weight Check ---")
    if invalid_teams.empty:
        print(f"✅ All {label} team weights sum to 1.0 (within buffer).")
    else:
        print(f"❌ Warning! The following teams have invalid sums:")
        print(invalid_teams)



# Run the checks
check_weights(merged_offense_clean, 'Player_Weight', 'Offense')
check_weights(merged_pitching_clean, 'P_Player_Weight', 'Pitching')

Length of Merged Offense:  794
Team: 0
Name: 0
Pos: 0
Status: 0
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 2
Is40Man: 0
Options: 257
Proj_PA: 108
Proj_IP: 791
Proj_PT: 108
first_name: 0
last_name: 0
player_id: 0
full_name: 0
position_code: 0
position_name: 0
position_abbr: 0
team_id: 0
team_name: 0
league_id: 0
league: 0
num_teams: 0
gamesPlayed: 0
runs: 0
doubles: 0
triples: 0
homeRuns: 0
baseOnBalls: 0
hits: 0
hitByPitch: 0
avg: 0
atBats: 0
obp: 0
slg: 0
ops: 0
stolenBases: 0
plateAppearances: 0
totalBases: 0
rbi: 0
sacBunts: 0
atBatsPerHomeRun: 0
strikeOuts: 0
intentionalWalks: 0
caughtStealing: 0
stolenBasePercentage: 0
caughtStealingPercentage: 0
groundIntoDoublePlay: 0
sacFlies: 0
babip: 0
groundOuts: 0
airOuts: 0
numberOfPitches: 0
leftOnBase: 0
groundOutsToAirouts: 0
catchersInterference: 0
first_name: 0
last_name: 0
gamesStarted: 0
assists: 0
putOuts: 0
errors: 0
chances: 0
fielding: 0
rangeFactorPerGame: 0
rangeFactorPer9Inn: 0
innings: 0
games: 0
doublePlays: 0
triplePla

,passedBall,catcherERA,wildPitches,pickoffs,Name,position_name
8,7.0,5.79,66.0,2.0,Gabriel Moreno,Catcher
13,38.0,4.14,290.0,10.0,James McCann,Catcher
20,0.0,5.66,7.0,1.0,Adrian Del Castillo,Catcher
21,4.0,6.5,40.0,1.0,Aramis Garcia,Catcher
27,2.0,5.71,7.0,0.0,Tyler Soderstrom,First Base
...,...,...,...,...,...,...
770,18.0,4.82,149.0,12.0,Keibert Ruiz,Catcher
775,1.0,6.02,20.0,0.0,Drew Millas,Catcher
778,0.0,3.86,1.0,0.0,Harry Ford,Catcher
781,10.0,4.43,80.0,0.0,Riley Adams,Catcher


Catcher/First Basemen Stats filled with zero if null

Team: 0
Name: 0
Pos: 0
Status: 0
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 2
Is40Man: 0
Options: 257
Proj_PA: 108
Proj_IP: 791
Proj_PT: 108
first_name: 0
last_name: 0
player_id: 0
full_name: 0
position_code: 0
position_name: 0
position_abbr: 0
team_id: 0
team_name: 0
league_id: 0
league: 0
num_teams: 0
gamesPlayed: 0
runs: 0
doubles: 0
triples: 0
homeRuns: 0
baseOnBalls: 0
hits: 0
hitByPitch: 0
avg: 0
atBats: 0
obp: 0
slg: 0
ops: 0
stolenBases: 0
plateAppearances: 0
totalBases: 0
rbi: 0
sacBunts: 0
atBatsPerHomeRun: 0
strikeOuts: 0
intentionalWalks: 0
caughtStealing: 0
stolenBasePercentage: 0
caughtStealingPercentage: 0
groundIntoDoublePlay: 0
sacFlies: 0
babip: 0
groundOuts: 0
airOuts: 0
numberOfPitches: 0
leftOnBase: 0
groundOutsToAirouts: 0
catchersInterference: 0
first_name: 0
last_name: 0
gamesStarted: 0
assists: 0
putOuts: 0
errors: 0
chances: 0
fielding: 0
rangeFactorPerGame: 0
rangeFactorPer9Inn: 0
innings: 0
games: 0
d

,Name,Team,Proj_PA,ServiceTime,Is40Man,Status
21,Aramis Garcia,ARI,NaN,3.060,N,NRI
22,Luken Baker,ARI,NaN,0.164,N,NRI
23,Óscar Mercado,ARI,NaN,3.031,N,NRI
24,Jacob Amaya,ARI,NaN,0.118,N,NRI
39,Cade Marlowe,ATH,NaN,0.090,N,NRI
...,...,...,...,...,...,...
740,Andrew Velazquez,TEX,NaN,3.033,N,NRI
741,Richie Martin,TEX,NaN,2.138,N,NRI
762,Carlos Mendoza,TOR,NaN,NaN,N,NRI
783,Tres Barrera,WSN,NaN,1.032,N,NRI


Offense Nulls Remaining:
Proj_PT    0
Proj_PA    0
Proj_IP    0
dtype: int64
Verfify Sums add to 1
Pitching
Team
ARI    1.0
ATH    1.0
ATL    1.0
BAL    1.0
BOS    1.0
Name: P_Player_Weight, dtype: float64
Offense
Team
ARI    1.0
ATH    1.0
ATL    1.0
BAL    1.0
BOS    1.0
Name: Player_Weight, dtype: float64
--- Offense Weight Check ---
✅ All Offense team weights sum to 1.0 (within buffer).
--- Pitching Weight Check ---
✅ All Pitching team weights sum to 1.0 (within buffer).


## Null Counts

In [122]:
print('Length of Merged Pitching: ', len(merged_pitching_clean))
for key, value in  merged_pitching_clean.isnull().sum().items():
    print(f"{key}: {value}")
print()
for key, value in  merged_offense_clean.isnull().sum().items():
    print(f"{key}: {value}")

Length of Merged Pitching:  815
Team: 0
Name: 0
Pos: 0
Status: 2
ProjectedOpeningDayRole: 0
Age: 0
ServiceTime: 0
Is40Man: 0
Options: 217
Proj_PA: 814
Proj_IP: 0
Proj_PT: 142
P_first_name: 0
P_last_name: 0
P_player_id: 0
P_full_name: 0
P_position_code: 0
P_position_name: 0
P_position_abbr: 0
P_team_id: 0
P_team_name: 0
P_league_id: 0
P_league: 0
P_num_teams: 0
P_gamesPlayed: 0
P_gamesStarted: 0
P_groundOuts: 0
P_airOuts: 0
P_runs: 0
P_doubles: 0
P_triples: 0
P_homeRuns: 0
P_strikeOuts: 0
P_baseOnBalls: 0
P_intentionalWalks: 0
P_hits: 0
P_hitByPitch: 0
P_avg: 0
P_atBats: 0
P_obp: 0
P_slg: 0
P_ops: 0
P_caughtStealing: 0
P_stolenBases: 0
P_stolenBasePercentage: 0
P_caughtStealingPercentage: 0
P_groundIntoDoublePlay: 0
P_numberOfPitches: 0
P_era: 0
P_inningsPitched: 0
P_wins: 0
P_losses: 0
P_saves: 0
P_saveOpportunities: 0
P_holds: 0
P_blownSaves: 0
P_earnedRuns: 0
P_whip: 0
P_battersFaced: 0
P_outs: 0
P_gamesPitched: 0
P_completeGames: 0
P_shutouts: 0
P_strikes: 0
P_strikePercentage: 0
P_

## Load Training Data
- Compare columsn of training data to the player career data

In [124]:
team_offense=pd.read_csv(os.path.join(data_dir,'TeamWins_clean.csv'))
team_fielding=pd.read_csv(os.path.join(data_dir,'teamFielding1995-2025.csv'))
league_info=pd.read_csv(os.path.join(data_dir,'leagueInfo.csv'))
league_info=league_info[['CODE','division_id','league_id']]
league_info['CODE']=league_info['CODE'].apply(lambda x: 'ATH' if x=='OAK' else x)
merged_offense_clean=pd.merge(merged_offense_clean.drop(columns=['league_id']),league_info, left_on='Team', right_on='CODE')
merged_pitching_clean=pd.merge(merged_pitching_clean.drop(columns=['P_league_id']),league_info, left_on='Team', right_on='CODE')

final_wins=pd.merge(team_offense, team_fielding.drop(columns=['LEAGUE', 'DIVISION', 'WINS','G']), on=['TEAM','Year'])
print('Columns in Training Data: ', final_wins.columns)
print()
print('Columns in 2026 Offensive Data: ', merged_offense_clean.columns)
print()
print('Columns in 2026 Pitching Data: ', merged_pitching_clean.columns)


Columns in Training Data:  Index(['TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
       'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
       'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
       'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
       'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
       'assists', 'chances', 'doublePlays', 'triplePlays',
       'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
       'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
       'stolenBasePercentage', 'catchersInterference', 'pickoffs'],
      dtype='object')

Columns in 2026 Offensive Data:  Index(['Team', 'Name', 'Pos', 'Status', 'ProjectedOpeningDayRole', 'Age',
       'ServiceTime', 'Is40Man', 'Options', 'Proj_PA', 'Proj_IP', 'Proj_PT',
       'first_name', 'last_name

## Column Renaming — Align Inference Data to Training Schema
Rename columns in the 2026 offense and pitching dataframes to match the
column names used in the historical training data.

In [126]:
# Hitting inference → training
offense_to_training = {
    'Team':               'TEAM',
    'league_id':          'LEAGUE',
    'division_id':        'DIVISION',
    'gamesPlayed':        'G',
    'atBats':             'AB',
    'runs':               'R',
    'hits':               'H',
    'doubles':            '2B',
    'triples':            '3B',
    'homeRuns':           'HR',
    'rbi':                'RBI',
    'baseOnBalls':        'BB',
    'strikeOuts':         'SO',
    'stolenBases':        'SB',
    'caughtStealing':     'CS',
    'avg':                'AVG',
    'obp':                'OBP',
    'slg':                'SLG',
    'ops':                'OPS',
    'fielding':           'fielding_pct',          # inference 'fielding' → training 'fielding_pct'
    'errors':             'errors',
    'putOuts':            'putOuts',
    'assists':            'assists',
    'chances':            'chances',
    'doublePlays':        'doublePlays',
    'triplePlays':        'triplePlays',
    'rangeFactorPerGame': 'rangeFactorPerGame',
    'rangeFactorPer9Inn': 'rangeFactorPer9Inn',
    'innings':            'innings',
    'passedBall':         'passedBall',
    'wildPitches':        'wildPitches',
    'throwingErrors':     'throwingErrors',
    'catchersInterference': 'catchersInterference',
    'stolenBasePercentage': 'stolenBasePercentage',
    'pickoffs':           'pickoffs',
}

# Pitching inference → training
pitching_to_training = {
    'Team':                 'TEAM',
    'league_id':            'LEAGUE',
    'division_id':          'DIVISION',
    'P_wins':               'W_pitch',
    'P_losses':             'L_pitch',
    'P_era':                'ERA_pitch',
    'P_gamesPitched':       'G_pitch',
    'P_gamesStarted':       'GS_pitch',
    'P_completeGames':      'CG_pitch',
    'P_shutouts':           'SHO_pitch',
    'P_saves':              'SV_pitch',
    'P_saveOpportunities':  'SVO_pitch',
    'P_inningsPitched':     'IP_pitch',
    'P_hits':               'H_pitch',
    'P_runs':               'R_pitch',
    'P_earnedRuns':         'ER_pitch',
    'P_homeRuns':           'HR_pitch',
    'P_hitBatsmen':         'HB_pitch',
    'P_baseOnBalls':        'BB_pitch',
    'P_strikeOuts':         'SO_pitch',
    'P_stolenBases':        'stolenBasesAllowed',   # stolen bases allowed by pitcher
    'P_whip':               'WHIP_pitch',
    'P_avg':                'AVG_pitch',
}

final_2026_offense = merged_offense_clean.rename(columns=offense_to_training)
final_2026_pitching = merged_pitching_clean.rename(columns=pitching_to_training)

## Derive Missing Hitting Columns for Training Data
Calculate columns present in the inference data but missing from the historical
training data. Exact calculations are always applied (`totalBases`, `stolenBasePercentage`,
`caughtStealingPercentage`, `atBatsPerHomeRun`). Approximate calculations (`babip`,
`plateAppearances`) are opt-in via `use_approx=True` since they rely on missing
components like sacrifice flies and hit-by-pitch.

In [128]:
def derive_hitting_columns(df,use_approx=False):
    df = df.copy()


    # Total bases: singles + 2x doubles + 3x triples + 4x HRs
    df['totalBases'] = (df['H'] - df['2B'] - df['3B'] - df['HR']) + \
                       (2 * df['2B']) + (3 * df['3B']) + (4 * df['HR'])

    # Stolen base percentage
    df['stolenBasePercentage'] = (df['SB'] / (df['SB'] + df['CS'])).replace([np.inf, -np.inf], 0).fillna(0)

    # Caught stealing percentage
    df['caughtStealingPercentage'] = (df['CS'] / (df['SB'] + df['CS'])).replace([np.inf, -np.inf], 0).fillna(0)

    # At bats per home run
    df['atBatsPerHomeRun'] = (df['AB'] / df['HR']).replace([np.inf, -np.inf], 0).fillna(0)

    if use_approx:
        # BABIP — approximate without sacFlies
        df['babip'] = ((df['H'] - df['HR']) / (df['AB'] - df['SO'] - df['HR'])).replace([np.inf, -np.inf], 0).fillna(0)
    
        # Plate appearances — approximate without HBP/SF/SH
        df['plateAppearances'] = df['AB'] + df['BB']

    return df

final_wins = derive_hitting_columns(final_wins)

## Training Data Column statistics for total type columns (RBI, Hits, Walks, etc)

In [130]:
tally_cols = ['G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'assists', 'putOuts', 'errors', 'chances', 'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall', 'wildPitches', 'pickoffs', 'catchersInterference']
league_stats = final_wins[tally_cols].mean().to_dict()
league_stats

{'G': 161.40044742729307,
 'AB': 5510.230425055928,
 'R': 743.3937360178971,
 'H': 1424.2494407158836,
 '2B': 282.18791946308727,
 '3B': 28.568232662192393,
 'HR': 175.03803131991052,
 'RBI': 707.8881431767338,
 'BB': 526.1040268456376,
 'SO': 1177.8557046979865,
 'SB': 96.92281879194631,
 'CS': 37.446308724832214,
 'assists': 1584.3366890380314,
 'putOuts': 4315.6767337807605,
 'errors': 100.04250559284117,
 'chances': 6000.055928411633,
 'doublePlays': 144.53691275167785,
 'triplePlays': 0.12639821029082773,
 'throwingErrors': 41.58612975391499,
 'passedBall': 10.885906040268456,
 'wildPitches': 53.70022371364653,
 'pickoffs': 1.9015659955257271,
 'catchersInterference': 1.1431767337807606}

## Team-Level Hitting & Fielding Projections (`aggregateHitting`)
Aggregate player-level 2026 stats to team-level projections using projected
plate appearances as weights. Tally stats are normalized per game, weighted,
summed by team, and scaled to a full 162-game season. Rate stats are weighted
averages only. All tally columns are mean-shifted to match historical league
averages before output. Final output is filtered to only the columns present
in the training data.

In [132]:
import pandas as pd
import numpy as np

class aggregateHitting:
    def __init__(self, df,historical_stats_dict):
        self.df = df.copy()
        self.historical_stats=historical_stats_dict
        self.training_columns=['TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
       'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
       'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
       'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
       'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
       'assists', 'chances', 'doublePlays', 'triplePlays',
       'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
       'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
       'stolenBasePercentage', 'catchersInterference', 'pickoffs',
       'totalBases', 'caughtStealingPercentage', 'atBatsPerHomeRun']
        
        self.hitting_tallies = [
                    'G',
                    'AB',
                    'R',
                    'H',
                    '2B',
                    '3B',
                    'HR',
                    'RBI',
                    'BB',
                    'SO',
                    'SB',
                    'CS',
                    'hitByPitch',
                    'intentionalWalks',
                    'plateAppearances',
                    'totalBases',
                    'sacBunts',
                    'sacFlies',
                    'groundIntoDoublePlay',
                    'groundOuts',
                    'airOuts',
                    'numberOfPitches',
                    'leftOnBase',
                    
                ]
        
        self.fielding_tallies = [
            'gamesStarted',
            'assists',
            'putOuts',
            'errors',
            'chances',
            'doublePlays',
            'triplePlays',
            'throwingErrors',
            'passedBall',
            'wildPitches',
            'pickoffs',
            'catchersInterference',
        ]
        
        self.rate_columns = [
            'AVG',
            'OBP',
            'SLG',
            'OPS',
            'babip',
            'stolenBasePercentage',
            'caughtStealingPercentage',
            'atBatsPerHomeRun',
            'groundOutsToAirouts',
            'fielding_pct',       # was 'fielding'
            'rangeFactorPerGame',
            'rangeFactorPer9Inn',
            'catcherERA',
            'stolenBasePercentage','atBatsPerHomeRun'
        ]
        
        self.all_tallies = self.hitting_tallies + self.fielding_tallies


        
        # --- CRITICAL FIX: Ensure types are numeric ---
        cols_to_fix = self.all_tallies + self.rate_columns + ['Player_Weight', 'G']
        for col in cols_to_fix:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce').fillna(0)
        # ----------------------------------------------

        self.add_per_game_columns()

    def add_per_game_columns(self):
        for col in self.all_tallies:
            if col in self.df.columns:
                # Divide by gamesPlayed (now guaranteed to be numeric)
                self.df[f"{col}_per_game"] = (self.df[col] / self.df['G']).replace([np.inf, -np.inf], 0).fillna(0)

    def calculate_team_projections(self, games_per_season=162, slots=10):
        # 1. Weight the Tally rates (this is the weighted sum part)
        for col in self.all_tallies:
            if f"{col}_per_game" in self.df.columns:
                # Player_Rate * Player_Weight
                self.df[f'w_tally_{col}'] = self.df[f"{col}_per_game"] * self.df['Player_Weight']
    
        # 2. Weight the Rate columns (Direct weighted average)
        for col in self.rate_columns:
            if col in self.df.columns:
                self.df[f'w_rate_{col}'] = self.df[col] * self.df['Player_Weight']
    
        # 3. Aggregation (Summing the weighted components)
        tally_cols = [c for c in self.df.columns if c.startswith('w_tally_')]
        rate_cols = [c for c in self.df.columns if c.startswith('w_rate_')]
        
        agg_map = {col: 'sum' for col in (tally_cols + rate_cols)}
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in self.df.columns:
                agg_map[cat] = 'first'
        
        team_df = self.df.groupby('TEAM').agg(agg_map)
    
        # 4. Scaling: 162 days * 9 slots * weighted_sum
        # This is equivalent to summing (Rate * Weight * 162) for 9 positions
        final_tallies = team_df[tally_cols] * games_per_season * slots
        final_tallies.columns = [c.replace('w_tally_', '') for c in final_tallies.columns]
        for col,avg in self.historical_stats.items():
            avg_diff=final_tallies[col].mean()-avg
            final_tallies[col]=final_tallies[col]-avg_diff
        
        # Rates: Just the weighted sum (the average skill of the lineup)
        final_rates = team_df[rate_cols]
        final_rates.columns = [c.replace('w_rate_', '') for c in final_rates.columns]
        
        result = pd.concat([final_tallies, final_rates], axis=1)
        
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in team_df.columns:
                result[cat] = team_df[cat]
        
        return result.reset_index()[['TEAM'] + [col for col in result.columns 
                                         if col in self.training_columns and col != 'TEAM']]
processor = aggregateHitting(final_2026_offense,league_stats)
team_2026_data = processor.calculate_team_projections()
team_2026_data

,TEAM,G,AB,R,H,2B,3B,HR,RBI,BB,...,SLG,OPS,stolenBasePercentage,caughtStealingPercentage,atBatsPerHomeRun,fielding_pct,rangeFactorPerGame,rangeFactorPer9Inn,LEAGUE,DIVISION
0,ARI,161.400447,5444.970146,764.162313,1424.908977,286.843678,43.641897,150.483963,703.316749,546.882938,...,0.413033,0.736386,0.736523,0.263651,32.163850,0.778079,3.170256,3.995190,NL,W
1,ATH,161.400447,5490.410137,716.586450,1440.464151,290.193550,21.671422,205.609311,715.603061,476.001303,...,0.441397,0.759820,0.725657,0.250474,29.748259,0.944101,3.828814,4.958342,AL,W
2,ATL,161.400447,5670.281007,820.789086,1502.702536,304.310589,26.856852,218.628553,790.481809,534.147214,...,0.447495,0.773563,0.670804,0.257909,26.785196,0.847437,3.729783,4.565561,NL,E
3,BAL,161.400447,5792.860505,805.023126,1437.876023,290.322642,32.649558,220.548592,768.418157,613.266393,...,0.430121,0.750833,0.680532,0.256172,24.873263,0.982965,4.074765,5.373804,AL,E
4,BOS,161.400447,5507.789408,771.676938,1456.565579,337.445097,31.788988,155.566688,689.325570,484.717739,...,0.428006,0.752435,0.685462,0.314538,34.107784,0.907154,3.482878,4.356681,AL,E
5,CHC,161.400447,5427.216289,781.630499,1416.329720,277.545761,35.029615,188.480560,772.440738,593.758955,...,0.437754,0.769166,0.696195,0.174199,28.190108,0.669414,2.525302,2.985808,NL,C
6,CHW,161.400447,5435.879491,691.369313,1386.945662,251.507681,15.888863,148.765935,647.199133,517.960853,...,0.394546,0.712164,0.628011,0.265885,42.448065,0.904382,3.923656,4.737519,AL,C
7,CIN,161.400447,5621.556619,781.398175,1427.268861,268.022896,31.401961,187.775042,703.341533,537.031976,...,0.420790,0.738345,0.669909,0.257217,29.558956,0.793848,2.880363,3.973995,NL,C
8,CLE,161.400447,5242.811930,695.547598,1297.928293,289.583704,26.873113,160.432064,663.764686,559.837271,...,0.407161,0.721222,0.752485,0.187674,34.223636,0.876589,3.400456,4.307478,AL,C
9,COL,161.400447,5392.271204,680.363854,1341.550552,254.378850,42.199897,138.929291,597.187002,421.951215,...,0.395135,0.699630,0.669273,0.267869,46.007473,0.842340,3.108581,4.090474,NL,W


## Derive Missing Pitching Rate Columns for Training Data
Calculate per-9-inning rates and ratio stats present in the inference data
but missing from the historical training data. Derived from existing `_pitch`
tally columns — strikeout/walk ratio, win percentage, and per-9 rates for
strikeouts, walks, hits, home runs, and runs scored.

In [134]:
def derive_pitching_columns(df):
    df = df.copy()
    
    # Ratio stats
    df['P_strikeoutWalkRatio'] = df['SO_pitch'] / df['BB_pitch'].replace(0, np.nan)
    
    # Win percentage
    df['P_winPercentage'] = df['W_pitch'] / (df['W_pitch'] + df['L_pitch']).replace(0, np.nan)
    
    # Save percentage
    df['SV_pct'] = df['SV_pitch'] / df['SVO_pitch'].replace(0, np.nan)
    
    # Per 9 inning rates
    ip = df['IP_pitch'].replace(0, np.nan)
    df['P_strikeoutsPer9Inn'] = (df['SO_pitch'] / ip) * 9
    df['P_walksPer9Inn']      = (df['BB_pitch'] / ip) * 9
    df['P_hitsPer9Inn']       = (df['H_pitch']  / ip) * 9
    df['P_homeRunsPer9']      = (df['HR_pitch'] / ip) * 9
    df['P_runsScoredPer9']    = (df['R_pitch']  / ip) * 9
    
    # Fill any infs from division
    rate_cols = ['P_strikeoutWalkRatio', 'P_winPercentage', 'SV_pct',
                 'P_strikeoutsPer9Inn', 'P_walksPer9Inn', 'P_hitsPer9Inn', 
                 'P_homeRunsPer9', 'P_runsScoredPer9']
    df[rate_cols] = df[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return df

final_wins = derive_pitching_columns(final_wins)
final_2026_pitching['SV_pct'] = final_2026_pitching['SV_pitch'] / final_2026_pitching['SVO_pitch'].replace(0, np.nan)

In [135]:
save_roles = final_2026_pitching[
    final_2026_pitching['ProjectedOpeningDayRole'].isin(['Closer', 'Setup Man'])
]

team_save_check = save_roles.groupby('TEAM').agg(
    n_closers   = ('ProjectedOpeningDayRole', lambda x: (x == 'Closer').sum()),
    n_setup_men = ('ProjectedOpeningDayRole', lambda x: (x == 'Setup Man').sum()),
    avg_SV_pct  = ('SV_pct', 'mean')
).reset_index()

team_save_check['has_closer_or_setup'] = (team_save_check['n_closers'] + team_save_check['n_setup_men']) > 0

print(f"Teams with no Closer or Setup Man: {(~team_save_check['has_closer_or_setup']).sum()}")
print(f"Teams with only Setup Men:         {((team_save_check['n_closers'] == 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print(f"Teams with only Closers:           {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] == 0)).sum()}")
print(f"Teams with both:                   {((team_save_check['n_closers'] > 0) & (team_save_check['n_setup_men'] > 0)).sum()}")
print()
display(team_save_check)

Teams with no Closer or Setup Man: 0
Teams with only Setup Men:         0
Teams with only Closers:           3
Teams with both:                   27



,TEAM,n_closers,n_setup_men,avg_SV_pct,has_closer_or_setup
0,ARI,2,2,0.528749,True
1,ATH,3,0,0.463768,True
2,ATL,1,2,0.610474,True
3,BAL,1,2,0.620899,True
4,BOS,1,2,0.598921,True
5,CHC,1,2,0.582011,True
6,CHW,1,2,0.674603,True
7,CIN,1,2,0.391667,True
8,CLE,1,2,0.481717,True
9,COL,1,2,0.444444,True


## Stats From pitching tally-like columns (Hits allowed, Earned Runs, etc)

In [137]:
starter_tallies = [
                            'GS_pitch',
                            'CG_pitch',
                            'SHO_pitch',
                            'IP_pitch',
                        ]

closer_tallies = [
            'SV_pitch',
            'SVO_pitch',
        ]
        
general_tallies = [
            'W_pitch',
            'L_pitch',
            'G_pitch',
            'H_pitch',
            'R_pitch',
            'ER_pitch',
            'HR_pitch',
            'HB_pitch',
            'BB_pitch',
            'SO_pitch',
            'P_strikeoutWalkRatio',
            'P_winPercentage',
            'P_strikeoutsPer9Inn',
            'P_walksPer9Inn',
            'P_hitsPer9Inn',
            'P_homeRunsPer9',
            'P_runsScoredPer9',
        ]
pitching_tallies = starter_tallies + closer_tallies + general_tallies

league_stats_pitch=final_wins[pitching_tallies].mean().to_dict()
league_stats_pitch

{'GS_pitch': 161.40044742729307,
 'CG_pitch': 4.837807606263982,
 'SHO_pitch': 9.256152125279643,
 'IP_pitch': 1438.3350111856826,
 'SV_pitch': 40.62751677852349,
 'SVO_pitch': 58.710290827740494,
 'W_pitch': 80.68791946308725,
 'L_pitch': 80.68791946308725,
 'G_pitch': 161.40044742729307,
 'H_pitch': 1424.2494407158836,
 'R_pitch': 743.3937360178971,
 'ER_pitch': 682.9619686800895,
 'HR_pitch': 175.03803131991052,
 'HB_pitch': 57.83221476510067,
 'BB_pitch': 526.1040268456376,
 'SO_pitch': 1177.8557046979865,
 'P_strikeoutWalkRatio': 2.2860244107256293,
 'P_winPercentage': 0.4999858647875603,
 'P_strikeoutsPer9Inn': 7.366116720654002,
 'P_walksPer9Inn': 3.2941064741807753,
 'P_hitsPer9Inn': 8.914879482010408,
 'P_homeRunsPer9': 1.0957146768999777,
 'P_runsScoredPer9': 4.655152472401818}

## Team-Level Pitching Projections (`aggregatePitching`)
Aggregate player-level 2026 pitching stats to team-level projections using
projected innings pitched as weights. Tally stats are normalized per inning,
weighted, summed by team, and scaled to a full 1458-inning season (9 × 162).
Rate stats (ERA, WHIP, AVG, per-9 rates) are weighted averages only. Save
columns (`SV`, `SVO`) are handled separately — 70% weight to the designated
closer and 30% split among setup men — then merged back into the final result.
All tally columns are mean-shifted to match historical league averages.

In [139]:
import warnings
class aggregatePitching:
    def __init__(self, df, historical_stats_dict):
        self.df = df.copy()
        self.historical_stats = historical_stats_dict
        self.training_columns = ['TEAM', 'LEAGUE', 'DIVISION', 'WINS', 'Year', 'G', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'RBI', 'BB', 'SO', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS',
       'W_pitch', 'L_pitch', 'ERA_pitch', 'G_pitch', 'GS_pitch', 'CG_pitch',
       'SHO_pitch', 'SV_pitch', 'SVO_pitch', 'IP_pitch', 'H_pitch', 'R_pitch',
       'ER_pitch', 'HR_pitch', 'HB_pitch', 'BB_pitch', 'SO_pitch',
       'WHIP_pitch', 'AVG_pitch', 'errors', 'fielding_pct', 'putOuts',
       'assists', 'chances', 'doublePlays', 'triplePlays',
       'rangeFactorPerGame', 'rangeFactorPer9Inn', 'innings', 'passedBall',
       'wildPitches', 'throwingErrors', 'caughtStealing', 'stolenBasesAllowed',
       'stolenBasePercentage', 'catchersInterference', 'pickoffs',
       'P_strikeoutWalkRatio', 'P_winPercentage', 'P_strikeoutsPer9Inn',
       'P_walksPer9Inn', 'P_hitsPer9Inn', 'P_homeRunsPer9',
       'P_runsScoredPer9','SV_pct']

        self.starter_tallies = [
                            'GS_pitch',
                            'CG_pitch',
                            'SHO_pitch',
                            'IP_pitch',
                        ]

        self.closer_tallies = [
            'SV_pitch',
            'SVO_pitch',
        ]
        self.closer_rates=['SV_pct']
        
        self.general_tallies = [
            'W_pitch',
            'L_pitch',
            'G_pitch',
            'H_pitch',
            'R_pitch',
            'ER_pitch',
            'HR_pitch',
            'HB_pitch',
            'BB_pitch',
            'SO_pitch'
        
        ]
        
        self.rate_columns = [
            'ERA_pitch',
            'WHIP_pitch',
            'AVG_pitch',
            'P_strikeoutWalkRatio',
            'P_winPercentage',
            'P_strikeoutsPer9Inn',
            'P_walksPer9Inn',
            'P_hitsPer9Inn',
            'P_homeRunsPer9',
            'P_runsScoredPer9',
        ]
        
        self.pitching_tallies = self.starter_tallies + self.closer_tallies + self.general_tallies
        self.all_tallies = self.pitching_tallies
        

        # Ensure numeric types
        cols_to_fix = self.all_tallies + self.rate_columns + ['Player_Weight', 'P_gamesPlayed']
        for col in cols_to_fix:
            if col in self.df.columns:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce').fillna(0)

        self.add_per_inning_columns()

    def add_per_inning_columns(self):
        for col in self.all_tallies:
            if col in self.df.columns:
                self.df[f"{col}_per_inning"] = (self.df[col] / self.df['IP_pitch']).replace([np.inf, -np.inf], 0).fillna(0)
    def calculate_team_projections(self, innings_per_season=1458):  # 9 * 162
        # 1. Weight the tally rates
        for col in self.all_tallies:
            if f"{col}_per_inning" in self.df.columns:
                self.df[f'w_tally_{col}'] = self.df[f"{col}_per_inning"] * self.df['P_Player_Weight']
    
        # 2. Weight the rate columns
        for col in self.rate_columns:
            if col in self.df.columns:
                self.df[f'w_rate_{col}'] = self.df[col] * self.df['P_Player_Weight']
    
        # 3. Aggregate by team
        tally_cols = [c for c in self.df.columns if c.startswith('w_tally_')]
        rate_cols  = [c for c in self.df.columns if c.startswith('w_rate_')]
    
        agg_map = {col: 'sum' for col in (tally_cols + rate_cols)}
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in self.df.columns:
                agg_map[cat] = 'first'
    
        team_df = self.df.groupby('TEAM').agg(agg_map)
    
        # 4. Scale tallies by total innings in a season
        final_tallies = team_df[tally_cols] * innings_per_season
        final_tallies.columns = [c.replace('w_tally_', '') for c in final_tallies.columns]
    
        # 5. Mean shift to match historical
        for col, avg in self.historical_stats.items():
            if col in final_tallies.columns:
                avg_diff = final_tallies[col].mean() - avg
                final_tallies[col] = final_tallies[col] - avg_diff
    
        # 6. Rates — just weighted sum
        final_rates = team_df[rate_cols]
        final_rates.columns = [c.replace('w_rate_', '') for c in final_rates.columns]
    
        result = pd.concat([final_tallies, final_rates], axis=1)
    
        for cat in ['LEAGUE', 'DIVISION']:
            if cat in team_df.columns:
                result[cat] = team_df[cat]
    
        return result.reset_index()[['TEAM'] + [col for col in result.columns 
                                         if col in self.training_columns and col != 'TEAM']]
    def calculate_closer_tallies(self, innings_per_season=1458):
        closer_roles = ['Closer', 'Setup Man']
        bullpen_df = self.df[self.df['ProjectedOpeningDayRole'].isin(closer_roles)].copy()
    
        def assign_save_weights(group):
            closers   = group['ProjectedOpeningDayRole'] == 'Closer'
            setup_men = group['ProjectedOpeningDayRole'] == 'Setup Man'
    
            n_closers   = closers.sum()
            n_setup_men = setup_men.sum()
    
            group = group.copy()
            group['save_weight'] = 0.0
    
            if n_closers > 0 and n_setup_men > 0:
                group.loc[closers,   'save_weight'] = 0.7 / n_closers
                group.loc[setup_men, 'save_weight'] = 0.3 / n_setup_men
            elif n_closers > 0:
                group.loc[closers,   'save_weight'] = 1.0 / n_closers
            elif n_setup_men > 0:
                group.loc[setup_men, 'save_weight'] = 1.0 / n_setup_men
    
            return group
    
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            bullpen_df = bullpen_df.groupby('TEAM', group_keys=False).apply(assign_save_weights)
           
        # Weight and aggregate closer tallies only
        for col in self.closer_tallies:
            if f"{col}_per_inning" in bullpen_df.columns:
                bullpen_df[f'w_tally_{col}'] = bullpen_df[f"{col}_per_inning"] * bullpen_df['save_weight']
    
        closer_tally_cols = [f'w_tally_{col}' for col in self.closer_tallies if f'w_tally_{col}' in bullpen_df.columns]
        closer_team = bullpen_df.groupby('TEAM')[closer_tally_cols].sum()
    
        final = closer_team * innings_per_season
        final.columns = [c.replace('w_tally_', '') for c in final.columns]
    
        # Mean shift
        for col, avg in self.historical_stats.items():
            if col in final.columns:
                avg_diff = final[col].mean() - avg
                final[col] = final[col] - avg_diff
    
        return final
    def calculate_closer_rates(self):
        closer_roles = ['Closer', 'Setup Man']
        bullpen_df = self.df[self.df['ProjectedOpeningDayRole'].isin(closer_roles)].copy()
    
        def assign_save_weights(group):
            closers   = group['ProjectedOpeningDayRole'] == 'Closer'
            setup_men = group['ProjectedOpeningDayRole'] == 'Setup Man'
            n_closers   = closers.sum()
            n_setup_men = setup_men.sum()
            group = group.copy()
            group['save_weight'] = 0.0
            if n_closers > 0 and n_setup_men > 0:
                group.loc[closers,   'save_weight'] = 0.7 / n_closers
                group.loc[setup_men, 'save_weight'] = 0.3 / n_setup_men
            elif n_closers > 0:
                group.loc[closers,   'save_weight'] = 1.0 / n_closers
            elif n_setup_men > 0:
                group.loc[setup_men, 'save_weight'] = 1.0 / n_setup_men
            return group
    
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            bullpen_df = bullpen_df.groupby('TEAM', group_keys=False).apply(assign_save_weights)
    
        # Weighted average of SV_pct using save_weight
        bullpen_df['w_SV_pct'] = bullpen_df['SV_pct'] * bullpen_df['save_weight']
        closer_rates = bullpen_df.groupby('TEAM')['w_SV_pct'].sum().rename('SV_pct')
    
        return closer_rates  # Series with TEAM as index

    def get_team_projections(self):
        result        = self.calculate_team_projections()   # TEAM is a column
        closer_result = self.calculate_closer_tallies()     # TEAM is index, SV_pitch/SVO_pitch
        closer_rates  = self.calculate_closer_rates()       # TEAM is index, SV_pct
    
        result = result.set_index('TEAM')
        result.update(closer_result)       # overwrite SV_pitch, SVO_pitch
        result['SV_pct'] = closer_rates    # add SV_pct
        result = result.reset_index()
    
        return result[['TEAM'] + [col for col in self.training_columns
                                   if col in result.columns and col != 'TEAM']]

processor = aggregatePitching(final_2026_pitching, league_stats_pitch)
team_2026_pitching = processor.get_team_projections()
team_2026_pitching

,TEAM,LEAGUE,DIVISION,W_pitch,L_pitch,ERA_pitch,G_pitch,GS_pitch,CG_pitch,SHO_pitch,...,WHIP_pitch,AVG_pitch,P_strikeoutWalkRatio,P_winPercentage,P_strikeoutsPer9Inn,P_walksPer9Inn,P_hitsPer9Inn,P_homeRunsPer9,P_runsScoredPer9,SV_pct
0,ARI,NL,W,83.339713,81.351188,4.052859,174.291455,152.465134,4.016893,8.974072,...,1.307135,0.246953,2.874498,0.500315,8.709557,3.271184,8.490885,1.071980,4.486298,0.501853
1,ATH,AL,W,87.305884,79.801157,4.139505,114.702149,165.205912,4.401225,9.181792,...,1.287765,0.238552,2.692469,0.535993,8.791967,3.477579,8.119876,1.283707,4.471829,0.463768
2,ATL,NL,E,85.138147,73.638036,3.938846,215.274539,140.257448,5.694245,9.189761,...,1.264207,0.236195,3.180319,0.520089,9.275535,3.373287,7.997230,1.045619,4.207314,0.752892
3,BAL,AL,E,77.797533,83.776288,4.239919,60.682982,183.284250,5.136824,9.470305,...,1.301645,0.246356,2.941059,0.468292,8.867988,3.202000,8.507601,1.200118,4.574542,0.716904
4,BOS,AL,E,81.351166,86.199920,3.768527,100.406711,171.188235,6.165283,10.407734,...,1.259188,0.235031,3.111780,0.484679,9.490772,3.326394,8.002055,0.992075,4.182664,0.753567
5,CHC,NL,C,84.282613,72.452409,3.997141,157.441953,162.867464,4.457477,8.915105,...,1.256268,0.240897,3.082468,0.555661,8.693946,3.074374,8.234624,1.249511,4.352902,0.730423
6,CHW,AL,C,66.610743,92.661003,4.302001,172.196333,153.788645,4.036464,8.993642,...,1.359947,0.240952,2.216521,0.466842,8.752558,4.014374,8.215282,1.107449,4.677502,0.670238
7,CIN,NL,C,74.377631,95.323671,4.236935,143.660533,185.669558,5.395961,9.843240,...,1.334906,0.239902,2.813717,0.462968,9.491633,3.686063,8.323748,1.243406,4.552785,0.520000
8,CLE,AL,C,79.842712,75.646269,3.882375,190.100212,166.583989,4.402829,9.009453,...,1.271868,0.234814,3.116270,0.523431,9.137829,3.437039,8.019237,1.054257,4.172651,0.590773
9,COL,NL,W,64.268273,104.630258,4.780300,126.498119,179.689193,4.093355,8.938012,...,1.414843,0.264759,2.318063,0.409353,7.565921,3.499400,9.236814,1.311166,5.182538,0.548333


# Save Processed Data To CSV

In [141]:
train_columns=final_wins.columns
print('2026 Pitching Columns')
print(team_2026_pitching.columns)
print()
print('2026 Hitting Columns')
print(team_2026_data.columns)


team_2026 = pd.merge(team_2026_data, team_2026_pitching, on=['TEAM', 'LEAGUE', 'DIVISION'])
print()
print('Final Concatenated Columns')


team_2026['Year']=2026
team_2026['WINS']=np.nan
print(team_2026.columns)
missing_cols=['innings', 'caughtStealing', 'stolenBasesAllowed']
final_wins=final_wins[[col for col in final_wins.columns if col not in missing_cols]]
team_2026=team_2026[[col for col in final_wins.columns]]
team_2026.to_csv(os.path.join(data_dir,'Processed_2026_Team_Data.csv'),index=False)

print()
print('Final Wins (training data) Columns')
final_wins.to_csv(os.path.join(data_dir,'Processed_Historical_Team_Wins_Data.csv'),index=False)
print(final_wins.columns)
A=final_wins.columns
B=team_2026.columns
X=[col for col in A if col not in B]
Y=[col for col in B if col not in A]
if len(X)==0 and len(Y)==0:
    print('\nColumns in training and inference are a perfect match')
else:
    print('\nColumns are not perfect match')


2026 Pitching Columns
Index(['TEAM', 'LEAGUE', 'DIVISION', 'W_pitch', 'L_pitch', 'ERA_pitch',
       'G_pitch', 'GS_pitch', 'CG_pitch', 'SHO_pitch', 'SV_pitch', 'SVO_pitch',
       'IP_pitch', 'H_pitch', 'R_pitch', 'ER_pitch', 'HR_pitch', 'HB_pitch',
       'BB_pitch', 'SO_pitch', 'WHIP_pitch', 'AVG_pitch',
       'P_strikeoutWalkRatio', 'P_winPercentage', 'P_strikeoutsPer9Inn',
       'P_walksPer9Inn', 'P_hitsPer9Inn', 'P_homeRunsPer9', 'P_runsScoredPer9',
       'SV_pct'],
      dtype='object')

2026 Hitting Columns
Index(['TEAM', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'SO', 'SB',
       'CS', 'totalBases', 'assists', 'putOuts', 'errors', 'chances',
       'doublePlays', 'triplePlays', 'throwingErrors', 'passedBall',
       'wildPitches', 'pickoffs', 'catchersInterference', 'AVG', 'OBP', 'SLG',
       'OPS', 'stolenBasePercentage', 'caughtStealingPercentage',
       'atBatsPerHomeRun', 'fielding_pct', 'rangeFactorPerGame',
       'rangeFactorPer9Inn', 'LEAGUE', 'DIVISION